# Detecting Malicious URLs with Lexical Features
**Arturo Gourentchik, Lachlan Carlsen, Mark Lester Apuya**  
Department of Computer Science – San Diego State University  
CS 549: Machine Learning. 
{agourentchik7972, lcarlsen7906, mapuya5582}@sdsu.edu  

## Section 1: Import Libraries
Import all necessary Python libraries for data manipulation, URL parsing, machine learning, model evaluation, and visualization.

In [ ]:
import pandas as pd
import numpy as np

from urllib.parse import urlparse
import tldextract

from scipy.stats import entropy
from collections import Counter
import math

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    train_test_split,
    GroupKFold,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns

import time

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

## Section 2: Load Data & Overiew 
Load the malicious/benign URL dataset and perform initial exploration to understand its structure, size, and data types.

### 2.1 Load Dataset
Load the CSV dataset containing URLs and their labels (benign=0, malicious=1).

In [ ]:
# Load the dataset
df = pd.read_csv('urldata.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 10 rows:")
df.head(10)

### 2.2 Exploratory Data Analysis
Examine dataset statistics, check for missing values, and understand the distribution of features.

In [ ]:
# Basic statistics
from IPython.display import display, Markdown, HTML
display(Markdown("## DATASET OVERVIEW"))

print(f"\nTotal URLs: {len(df):,}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nDuplicate Rows: {df.duplicated().sum():,}")

# Class distribution
from IPython.display import display, Markdown, HTML
display(Markdown("## CLASS DISTRIBUTION"))

print(f"\nLabel Counts:\n{df['label'].value_counts()}")
print(f"\nLabel Proportions:\n{df['label'].value_counts(normalize=True)}")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#2ecc71', '#e74c3c']  # Green for benign, Red for malicious
counts = df['label'].value_counts().sort_index()
bars = axes[0].bar(['Benign', 'Malicious'], counts.values, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_ylim(0, counts.max() * 1.15)

# Add count labels on bars
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.02, 
                 f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Benign', 'Malicious'], colors=colors, autopct='%1.1f%%',
            startangle=90, explode=(0.02, 0.02), shadow=True,
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Benign vs Malicious URLs', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.3 Sample URLs and Labels
Display example URLs from both benign and malicious classes to understand the data patterns.

In [ ]:
# Sample benign URLs
from IPython.display import display, Markdown, HTML
display(Markdown("## SAMPLE BENIGN URLs"))
benign_urls = df[df['result'] == 0]['url'].sample(5, random_state=42)
for url in benign_urls:
    print(url)

# Sample malicious URLs
from IPython.display import display, Markdown, HTML
display(Markdown("## SAMPLE MALICIOUS URLs"))
malicious_urls = df[df['result'] == 1]['url'].sample(5, random_state=42)
for url in malicious_urls:
    print(url)

# PRELIMINARY LEXICAL PATTERN ANALYSIS
from IPython.display import display, Markdown, HTML
display(Markdown("## PRELIMINARY LEXICAL PATTERN ANALYSIS"))

# Create temporary features for exploration
temp_df = df.copy()
temp_df['url_length'] = temp_df['url'].apply(len)
temp_df['num_dots'] = temp_df['url'].apply(lambda x: x.count('.'))
temp_df['num_slashes'] = temp_df['url'].apply(lambda x: x.count('/'))
temp_df['num_digits'] = temp_df['url'].apply(lambda x: sum(c.isdigit() for c in x))
temp_df['num_special'] = temp_df['url'].apply(lambda x: sum(c in '-_@?&=%#' for c in x))
temp_df['has_https'] = temp_df['url'].apply(lambda x: 1 if x.startswith('https') else 0)
temp_df['has_ip'] = temp_df['url'].apply(lambda x: 1 if any(c.isdigit() and '.' in x[max(0,x.find(c)-3):x.find(c)+4] for c in x) else 0)

# Compare means by class
features = ['url_length', 'num_dots', 'num_slashes', 'num_digits', 'num_special', 'has_https']
comparison = temp_df.groupby('result')[features].mean().T
comparison.columns = ['Benign', 'Malicious']
comparison['Difference'] = comparison['Malicious'] - comparison['Benign']
print("\nMean Feature Values by Class:")
print(comparison.round(2))

# HTTPS usage comparison
from IPython.display import display, Markdown, HTML
display(Markdown("## HTTPS USAGE BY CLASS"))
https_stats = temp_df.groupby('result')['has_https'].mean() * 100
print(f"Benign URLs using HTTPS: {https_stats[0]:.1f}%")
print(f"Malicious URLs using HTTPS: {https_stats[1]:.1f}%")

# www. usage comparison
from IPython.display import display, Markdown, HTML
display(Markdown("## www. USAGE BY CLASS"))
temp_df['has_www'] = temp_df['url'].apply(lambda x: 1 if 'www.' in x else 0)
www_stats = temp_df.groupby('result')['has_www'].mean() * 100
print(f"Benign URLs using www: {www_stats[0]:.1f}%")
print(f"Malicious URLs using www: {www_stats[1]:.1f}%")

del temp_df

## Section 3: Data Preprocessing
Clean, normalize, and transform raw URLs into a structured feature set suitable for machine learning models.

### 3.1 Handling Missing or Invalid URLs
Identify and remove records with missing, empty, or unparseable URL strings.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.1 HANDLING MISSING OR INVALID URLs"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")
print(f"[INPUT] Sample:\n{df[['url', 'label', 'result']].head(3)}")

# Track initial count
initial_count = len(df)

# Remove rows with missing URLs
df = df.dropna(subset=['url'])

# Remove empty strings
df = df[df['url'].str.strip() != '']

# Remove URLs that can't be parsed
def is_valid_url(url):
    try:
        result = urlparse(url)
        return all([result.scheme or url.startswith('www'), len(url) > 5])
    except:
        return False

df = df[df['url'].apply(is_valid_url)]

# OUTPUT
removed_count = initial_count - len(df)
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] Removed: {removed_count:,} invalid/missing URLs")
print(f"[OUTPUT] Remaining: {len(df):,} valid URLs")

### 3.2 Remove Duplicate URLs
Eliminate duplicate URLs to prevent data leakage and training bias.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.2 REMOVE DUPLICATE URLs"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")
print(f"[INPUT] Duplicate count: {df.duplicated(subset=['url']).sum():,}")

# Track initial count
initial_count = len(df)

# Remove exact duplicate URLs (keep first occurrence)
df = df.drop_duplicates(subset=['url'], keep='first')

# Reset index
df = df.reset_index(drop=True)

# OUTPUT
removed_count = initial_count - len(df)
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] Removed: {removed_count:,} duplicate URLs")
print(f"[OUTPUT] Remaining: {len(df):,} unique URLs")

### 3.3 URL Normalization 
Standardize URLs by lowercasing, removing fragments, stripping default ports, and collapsing repeated slashes.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.3 URL NORMALIZATION"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")
print(f"[INPUT] Sample URLs:")
for url in df['url'].head(3):
    print(f"  {url}")

def normalize_url(url):
    # Lowercase the URL
    url = url.lower()
    
    # Remove fragments
    if '#' in url:
        url = url.split('#')[0]
    
    # Remove trailing slashes
    url = url.rstrip('/')
    
    # Remove default ports (:80 for http, :443 for https)
    url = url.replace(':80/', '/').replace(':443/', '/')
    url = url.replace(':80', '').replace(':443', '')
    
    # Collapse multiple slashes
    if '://' in url:
        protocol, rest = url.split('://', 1)
        rest = '/'.join(filter(None, rest.split('/')))
        url = f"{protocol}://{rest}"
    
    return url

# Apply normalization
df['url_normalized'] = df['url'].apply(normalize_url)

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] New column added: 'url_normalized'")

Sample Modified URLs

In [ ]:
# Find modified URLs
modified_urls = df[df['url'] != df['url_normalized']]
print(f"[OUTPUT] URLs modified: {len(modified_urls):,} ({len(modified_urls)/len(df)*100:.2f}%)")

# Modified URLs
if len(modified_urls) > 0:
    print(f"\n[OUTPUT] Sample MODIFIED URLs:")
    for i, row in modified_urls[['url', 'url_normalized']].sample(min(5, len(modified_urls)), random_state=42).iterrows():
        print(f"  Original:   {row['url']}")
        print(f"  Normalized: {row['url_normalized']}\n")
else:
    print(f"\n[OUTPUT] No URLs required normalization changes.")
    print(f"[OUTPUT] Sample URLs (unchanged):")
    for i, row in df[['url', 'url_normalized']].sample(3, random_state=42).iterrows():
        print(f"  {row['url']}")

print(f"\nColumn Names: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 10 rows:")
df.head(10)

### 3.4 Lexical Feature Extraction
Extract numerical features from URL strings without fetching page content.

#### 3.4.1 Length Features
Calculate length-based features: total URL length, host length, path length, and query length.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.4.1 LENGTH FEATURES"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")
print(f"[INPUT] Columns: {df.columns.tolist()}")

def extract_length_features(url):
    """Extract length-based features from URL"""
    parsed = urlparse(url)
    
    return {
        'url_length': len(url),
        'host_length': len(parsed.netloc),
        'path_length': len(parsed.path),
        'query_length': len(parsed.query)
    }

# Apply feature extraction to all URLs
length_features = df['url_normalized'].apply(extract_length_features).apply(pd.Series)

# Add to dataframe
df = pd.concat([df, length_features], axis=1)

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] New features: {length_features.columns.tolist()}")
print(f"\n[OUTPUT] Features:")
print(df[['url_length', 'host_length', 'path_length', 'query_length']].describe().round(2))

#### 3.4.2 Count Features
Count occurrences of special characters, digits, subdomain levels, and path segments.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.4.2 COUNT FEATURES"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")

def extract_count_features(url):
    """Count occurrences of special characters and patterns"""
    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    return {
        'num_dots': url.count('.'), # of dots
        'num_hyphens': url.count('-'), # of hyphens
        'num_underscores': url.count('_'), # of underscores
        'num_slashes': url.count('/'), # of slashes
        'num_question_marks': url.count('?'), # of question marks
        'num_equals': url.count('='), # of equals signs
        'num_ampersands': url.count('&'), # of ampersands
        'num_at_symbols': url.count('@'), # of at symbols
        'num_digits': sum(c.isdigit() for c in url), # of digits
        'num_letters': sum(c.isalpha() for c in url), # of letters
        'num_subdomains': len(ext.subdomain.split('.')) if ext.subdomain else 0, # of subdomains
        'num_path_segments': len([s for s in parsed.path.split('/') if s]), # of path segments
        'num_params': len(parsed.query.split('&')) if parsed.query else 0 # of query parameters
        # ADD more count features as needed
    }   

# Apply feature extraction to all URLs
count_features = df['url_normalized'].apply(extract_count_features).apply(pd.Series)

# Add to dataframe
df = pd.concat([df, count_features], axis=1)

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] New features: {count_features.columns.tolist()}")
print(f"\n[OUTPUT] Feature:")
print(count_features.describe().round(2))

#### 3.4.3 Ratio Features
Compute ratios: digits-to-length, symbols-to-length, uppercase-to-lowercase.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.4.3 RATIO FEATURES"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")

# Calculate ratio features (avoid division by zero)
df['digit_ratio'] = df['num_digits'] / df['url_length'].replace(0, 1) # of digits / url length
df['letter_ratio'] = df['num_letters'] / df['url_length'].replace(0, 1) # of letters / url length
df['symbol_ratio'] = (df['num_hyphens'] + df['num_underscores'] + df['num_at_symbols']) / df['url_length'].replace(0, 1) # of symbols / url length
df['digit_letter_ratio'] = df['num_digits'] / df['num_letters'].replace(0, 1) # of digits / letters
df['path_url_ratio'] = df['path_length'] / df['url_length'].replace(0, 1) # path length / url length
df['query_url_ratio'] = df['query_length'] / df['url_length'].replace(0, 1) # query length / url length
df['host_url_ratio'] = df['host_length'] / df['url_length'].replace(0, 1) # host length / url length
# Add more ratio features as needed

ratio_features = ['digit_ratio', 'letter_ratio', 'symbol_ratio', 'digit_letter_ratio', 
                  'path_url_ratio', 'query_url_ratio', 'host_url_ratio']

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] New features: {ratio_features}")
print(f"\n[OUTPUT] Features:")
print(df[ratio_features].describe().round(4))

#### 3.4.4 Entropy Features
Calculate Shannon entropy for the full URL and path to measure randomness/complexity.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.4.4 ENTROPY FEATURES"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")

def calculate_entropy(text):
    """Calculate Shannon entropy of a string"""
    if not text:
        return 0.0
    freq = Counter(text)
    probs = [count / len(text) for count in freq.values()]
    return entropy(probs, base=2) # Shannon entropy

def extract_entropy_features(url):
    """Extract entropy-based features"""
    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    return {
        'url_entropy': calculate_entropy(url), # entropy of entire URL
        'host_entropy': calculate_entropy(parsed.netloc), # entropy of host
        'path_entropy': calculate_entropy(parsed.path), # entropy of path
        'domain_entropy': calculate_entropy(ext.domain) # entropy of domain
    }

# Apply feature extraction to all URLs
entropy_features = df['url_normalized'].apply(extract_entropy_features).apply(pd.Series)

# Add to dataframe
df = pd.concat([df, entropy_features], axis=1)

entropy_cols = ['url_entropy', 'host_entropy', 'path_entropy', 'domain_entropy']

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] New features: {entropy_cols}")
print(f"\n[OUTPUT] Features:")
print(df[entropy_cols].describe().round(4))

#### 3.4.5 Binary Flag Features
Create binary indicators: HTTPS presence, IP address in host, suspicious extensions, and keyword flags.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.4.5 BINARY FLAG FEATURES"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")

import re

def extract_binary_features(url):
    """Extract binary indicator features"""
    parsed = urlparse(url)
    
    # IP address pattern
    ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}' # IPv4 pattern 
    
    # Suspicious keywords
    suspicious_words = ['secure', 'login', 'signin', 'bank', 'account', 'update', 
                        'verify', 'password', 'confirm', 'suspend', 'alert'] # common phishing words... Add more as needed
    
    # Suspicious extensions
    suspicious_ext = ['.exe', '.php', '.js', '.bat', '.cmd', '.scr'] # common executable/script extensions... Add more as needed
    
    # URL shorteners
    shorteners = ['bit.ly', 'tinyurl', 't.co', 'goo.gl', 'ow.ly', 'is.gd', 'buff.ly'] # common URL shorteners... Add more as needed

    # ADD more binary features as needed
    
    return {
        'has_https': 1 if url.startswith('https') else 0, # of https
        'has_http': 1 if url.startswith('http://') else 0, # of http
        'has_ip_address': 1 if re.search(ip_pattern, parsed.netloc) else 0, # presence of IP address
        'has_at_symbol': 1 if '@' in url else 0, # presence of '@' symbol
        'has_port': 1 if ':' in parsed.netloc and not parsed.netloc.startswith('[') else 0, # presence of port number
        'has_suspicious_word': 1 if any(word in url.lower() for word in suspicious_words) else 0, # presence of suspicious words
        'has_suspicious_ext': 1 if any(ext in url.lower() for ext in suspicious_ext) else 0, # presence of suspicious extensions
        'is_shortened': 1 if any(shortener in url.lower() for shortener in shorteners) else 0, # is URL shortened
        'has_subdomain': 1 if tldextract.extract(url).subdomain else 0, # presence of subdomain
        'has_www': 1 if 'www.' in url.lower() else 0 # presence of www
        # Add more binary features as needed
    }

# Apply feature extraction to all URLs
binary_features = df['url_normalized'].apply(extract_binary_features).apply(pd.Series)

# Add to dataframe
df = pd.concat([df, binary_features], axis=1)

binary_cols = binary_features.columns.tolist()

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] New features: {binary_cols}")
print(f"\n[OUTPUT] Feature distribution:")
print(df[binary_cols].sum().to_frame('Count').T)
print(f"\n[OUTPUT] Percentage of URLs with each flag:")
print((df[binary_cols].mean() * 100).round(2).to_frame('%').T)

### 3.5 Categorical Feature Encoding
One-hot encode categorical features such as protocol scheme and TLD bucket.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.5 CATEGORICAL FEATURE ENCODING"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")

def extract_categorical_features(url):
    """Extract categorical features from URL"""
    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    # Protocol scheme
    scheme = parsed.scheme if parsed.scheme else 'none' # https, http, ftp...
    
    # TLD
    tld = ext.suffix if ext.suffix else 'unknown' # com, org, net, co.uk...
    
    return {
        'scheme': scheme,
        'tld': tld
    }

# Extract categorical features
cat_features = df['url_normalized'].apply(extract_categorical_features).apply(pd.Series)

# Get TLD frequency and bucket rare TLDs
tld_counts = cat_features['tld'].value_counts()
common_tlds = tld_counts[tld_counts >= 100].index.tolist()  # TLDs with 100+ occurrences
cat_features['tld_bucket'] = cat_features['tld'].apply(lambda x: x if x in common_tlds else 'other')

# One-hot encode scheme
scheme_dummies = pd.get_dummies(cat_features['scheme'], prefix='scheme')

# One-hot encode TLD bucket
tld_dummies = pd.get_dummies(cat_features['tld_bucket'], prefix='tld')

# Add to dataframe
df = pd.concat([df, scheme_dummies, tld_dummies], axis=1)

# Store categorical column names
scheme_cols = scheme_dummies.columns.tolist()
tld_cols = tld_dummies.columns.tolist()
categorical_cols = scheme_cols + tld_cols

# OUTPUT
print(f"\n[OUTPUT] Shape: {df.shape}")
print(f"[OUTPUT] Scheme features: {scheme_cols}")
print(f"[OUTPUT] TLD features ({len(tld_cols)}): {tld_cols[:10]}...")
print(f"\n[OUTPUT] Scheme distribution:")
print(cat_features['scheme'].value_counts())
print(f"\n[OUTPUT] Top 10 TLD buckets:")
print(cat_features['tld_bucket'].value_counts().head(10))

### 3.6 Feature Scaling for Linear Models
Apply StandardScaler (z-score normalization) to continuous features for linear models. Tree-based models use unscaled features.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.6 FEATURE SCALING PREPARATION"))

# INPUT
print(f"\n[INPUT] Shape: {df.shape}")

# Define feature groups
length_cols = ['url_length', 'host_length', 'path_length', 'query_length'] # length features
count_cols = ['num_dots', 'num_hyphens', 'num_underscores', 'num_slashes', 
              'num_question_marks', 'num_equals', 'num_ampersands', 'num_at_symbols',
              'num_digits', 'num_letters', 'num_subdomains', 'num_path_segments', 'num_params'] # count features
ratio_cols = ['digit_ratio', 'letter_ratio', 'symbol_ratio', 'digit_letter_ratio',
              'path_url_ratio', 'query_url_ratio', 'host_url_ratio'] # ratio features
entropy_cols = ['url_entropy', 'host_entropy', 'path_entropy', 'domain_entropy'] # entropy features
binary_cols = ['has_https', 'has_http', 'has_ip_address', 'has_at_symbol', 
               'has_double_slash', 'has_port', 'has_suspicious_word', 
               'has_suspicious_ext', 'is_shortened', 'has_subdomain', 'has_www'] # binary features

# Continuous features
continuous_cols = length_cols + count_cols + ratio_cols + entropy_cols 

# All feature columns
all_feature_cols = continuous_cols + binary_cols + categorical_cols

print(f"\n[OUTPUT] Feature groups:")
print(f"  - Continuous features (to scale): {len(continuous_cols)}")
print(f"  - Binary features: {len(binary_cols)}")
print(f"  - Categorical (one-hot): {len(categorical_cols)}")
print(f"  - Total features: {len(all_feature_cols)}")

# Prepare feature matrix X and target y
X = df[all_feature_cols].copy()
y = df['result'].copy()

print(f"\n[OUTPUT] X shape: {X.shape}")
print(f"[OUTPUT] y shape: {y.shape}")
print(f"[OUTPUT] y distribution:\n{y.value_counts()}")

### 3.7 Handling Class Imbalance 
Compute class weights to handle imbalanced data during model training.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.7 HANDLING CLASS IMBALANCE"))

# INPUT
print(f"\n[INPUT] Class distribution:")
print(y.value_counts())
print(f"\n[INPUT] Class proportions:")
print(y.value_counts(normalize=True))

# Compute class weights
classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=y)
class_weights = {0: class_weights_array[0], 1: class_weights_array[1]}

# Calculate imbalance ratio
imbalance_ratio = y.value_counts()[0] / y.value_counts()[1]

# OUTPUT
print(f"\n[OUTPUT] Imbalance ratio: {imbalance_ratio:.2f}:1")
print(f"[OUTPUT] Computed class weights:")
print(f"  - Class 0 (Benign): {class_weights[0]:.4f}")
print(f"  - Class 1 (Malicious): {class_weights[1]:.4f}")
print(f"\n[OUTPUT] Weights that will be used during model training to handle imbalance.")

### 3.8 Domain-Aware Train/Validation/Test Split
Split data by registrable domain to prevent leakage from related URLs appearing in both train and test sets.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.8 DOMAIN-AWARE TRAIN/VALIDATION/TEST SPLIT"))

# INPUT
print(f"\n[INPUT] X shape: {X.shape}")
print(f"[INPUT] y shape: {y.shape}")

# Extract available domain for grouping
df['registrable_domain'] = df['url_normalized'].apply(
    lambda x: tldextract.extract(x).registered_domain
)

# Handle empty domains
df['registrable_domain'] = df['registrable_domain'].replace('', 'unknown_domain')

# Get unique domains
unique_domains = df['registrable_domain'].unique()
print(f"\n[INFO] Unique registrable domains: {len(unique_domains):,}")

# Create domain to group mapping
domain_to_group = {domain: idx for idx, domain in enumerate(unique_domains)}
groups = df['registrable_domain'].map(domain_to_group).values

# Split domains into train/val/test
np.random.seed(42)
shuffled_domains = np.random.permutation(unique_domains)

# 70% train, 15% validation, 15% test (by domain) 
n_domains = len(shuffled_domains)
train_end = int(0.70 * n_domains)
val_end = int(0.85 * n_domains)

train_domains = set(shuffled_domains[:train_end])
val_domains = set(shuffled_domains[train_end:val_end])
test_domains = set(shuffled_domains[val_end:])

# Create masks
train_mask = df['registrable_domain'].isin(train_domains)
val_mask = df['registrable_domain'].isin(val_domains)
test_mask = df['registrable_domain'].isin(test_domains)

# Split data
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Store groups for cross validation
groups_train = groups[train_mask]

# OUTPUT
print(f"\n[OUTPUT] Split by registrable domain to prevent data leakage:")
print(f"\n  Training set:")
print(f"    - Domains: {len(train_domains):,}")
print(f"    - Samples: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"    - Class distribution: {dict(y_train.value_counts())}")

print(f"\n  Validation set:")
print(f"    - Domains: {len(val_domains):,}")
print(f"    - Samples: {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)")
print(f"    - Class distribution: {dict(y_val.value_counts())}")

print(f"\n  Test set:")
print(f"    - Domains: {len(test_domains):,}")
print(f"    - Samples: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"    - Class distribution: {dict(y_test.value_counts())}")

# Verify no domain overlap
train_val_overlap = train_domains.intersection(val_domains)
train_test_overlap = train_domains.intersection(test_domains)
val_test_overlap = val_domains.intersection(test_domains)
print(f"\n[OUTPUT] Domain overlap verification:")
print(f"  - Train/Val overlap: {len(train_val_overlap)} domains")
print(f"  - Train/Test overlap: {len(train_test_overlap)} domains")
print(f"  - Val/Test overlap: {len(val_test_overlap)} domains")

### 3.9 Data Preprocessing output

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 3.9 DATA PREPROCESSING"))

# Summary
print(f"\n[FEATURES]")
print(f"  Total features: {len(all_feature_cols)}")
print(f"  - Length features: {len(length_cols)}")
print(f"  - Count features: {len(count_cols)}")
print(f"  - Ratio features: {len(ratio_cols)}")
print(f"  - Entropy features: {len(entropy_cols)}")
print(f"  - Binary features: {len(binary_cols)}")
print(f"  - Categorical (one-hot): {len(categorical_cols)}")

print(f"\n[DATASETS]")
print(f"  Training:   {X_train_scaled.shape[0]:,} samples")
print(f"  Validation: {X_val_scaled.shape[0]:,} samples")
print(f"  Test:       {X_test_scaled.shape[0]:,} samples")

print(f"\n[CLASS WEIGHTS]")
print(f"  Benign (0):    {class_weights[0]:.4f}")
print(f"  Malicious (1): {class_weights[1]:.4f}")

print(f"\n[SCALING]")
print(f"  StandardScaler applied to {len(continuous_cols)} continuous features")
print(f"  Scaler fit on training data only (no data leakage)")

print(f"\n[READY FOR MODELING]")
print(f"  - X_train_scaled, y_train (for linear models)")
print(f"  - X_train, y_train (for tree-based models)")
print(f"  - X_val_scaled, y_val (validation)")
print(f"  - X_test_scaled, y_test (final evaluation)")
print(f"  - groups_train (for GroupKFold CV)")
print(f"  - class_weights (for handling imbalance)")

# Display feature list
print(f"\n[FEATURE LIST]")
for i, col in enumerate(all_feature_cols, 1):
    print(f"  {i:2d}. {col}")

## Section 4: Model Pipelines

### 4.1 Logistic Regression
Linear baseline model with calibrated probabilities and interpretable feature coefficients.

In [ ]:
from IPython.display import display, Markdown, HTML
display(Markdown("## 4.1 LOGISTIC REGRESSION"))

# Initialize Logistic Regression with class weights
print("[STEP 1] Initialize Logistic Regression model")
lr_model = LogisticRegression(
    class_weight=class_weights,
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

# Train on scaled data
print("[STEP 2] Training on scaled training data...")
start_time = time.time()
lr_model.fit(X_train_scaled, y_train)
lr_train_time = time.time() - start_time
print(f"  Training time: {lr_train_time:.2f} seconds")

# Predict on validation set
print("[STEP 3] Generating predictions on validation set...")
lr_val_pred = lr_model.predict(X_val_scaled)
lr_val_proba = lr_model.predict_proba(X_val_scaled)[:, 1]

# Evaluate on validation set
print("\n[VALIDATION RESULTS]")
print(f"  Accuracy:  {accuracy_score(y_val, lr_val_pred):.4f}")
print(f"  Precision: {precision_score(y_val, lr_val_pred):.4f}")
print(f"  Recall:    {recall_score(y_val, lr_val_pred):.4f}")
print(f"  F1 Score:  {f1_score(y_val, lr_val_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, lr_val_proba):.4f}")
print(f"  PR-AUC:    {average_precision_score(y_val, lr_val_proba):.4f}")

# Display top feature coefficients
print("\n[TOP 10 FEATURE COEFFICIENTS]")
coef_df = pd.DataFrame({
    'feature': all_feature_cols,
    'coefficient': lr_model.coef_[0]
})
coef_df['abs_coef'] = np.abs(coef_df['coefficient'])
coef_df = coef_df.sort_values('abs_coef', ascending=False)

print("\n  Most influential features (by absolute coefficient):")
for i, row in coef_df.head(10).iterrows():
    direction = "+" if row['coefficient'] > 0 else "-"
    print(f"    {direction} {row['feature']}: {row['coefficient']:.4f}")

# Store model for later comparison
models = {'Logistic Regression': lr_model}
model_results = {
    'Logistic Regression': {
        'model': lr_model,
        'train_time': lr_train_time,
        'val_pred': lr_val_pred,
        'val_proba': lr_val_proba,
        'scaled': True  # Uses scaled features
    }
}

print("\n[OUTPUT] Logistic Regression model trained and stored.")
print(f"[OUTPUT] Model parameters: {lr_model.get_params()}")

### 4.2 Linear SVM
Margin-based classifier with probability calibration for reliable score outputs.

### 4.3 Random Forest
Ensemble tree model that captures non-linear patterns and feature interactions.

## Section 5: Hyperparameter Tuning (Grouped Cross-Validation)
Tune model hyperparameters using GroupKFold cross-validation to ensure domain-aware evaluation.

### 5.1 CV Setup
Configure GroupKFold with 5 folds, grouping by registrable domain to prevent data leakage.

### 5.2 Logistic Regression Tuning
Grid search over regularization parameter and loss function.

### 5.3 Linear SVM Tuning

### 5.4 Random Forest Tuning
Randomized search over n_estimators, max_depth, min_samples_split, and max_features.

## Section 6: Probability Valibration & Threshold Selection
Calibrate model probabilities and select optimal decision thresholds for deployment.

### 6.1 Probability Calibration

### 6.2 Threshold Optimization (Maximize F1)
Select the decision threshold on validation data that maximizes F1 score.

## Section 7: Model Evaluation
Evaluate trained models on the held-out test set using multiple performance metrics.

### 7.1 Performance Metrics
Calculate Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC for each model.

### 7.2 Confusion Matricies
Visualize true positives, true negatives, false positives, and false negatives for each model.

### 7.3 ROC Curves
Plot Receiver Operating Characteristic curves comparing all models.

### 7.4 Percision-Recall Curves
Plot Precision-Recall curves, especially important for imbalanced datasets.

### 7.5 Interference Latency Measurement
Measure prediction time per URL to ensure the model meets low-latency requirements.

## Section 8: Results & Comparative Analysis
Compare model performance and select the best model.

### 8.1 Model Comparison Table

### 8.2 Best Model Selection
Select the final model based on F1 score, ROC-AUC, and inference latency criteria.

### 8.3 Feature Importance Analysis
Analyze which lexical features contribute most to malicious URL detection.